# Ejercicio 8 – Pedido con entrega (Tkinter)

Reto integrador de la Clase 3 de Tkinter: formulario de pedidos de una farmacia con retiro en local o entrega a domicilio. Los datos y precios son ficticios.

## Qué hace

- Registra cliente (nombre y teléfono), producto, precio unitario y cantidad.
- Permite aplicar un descuento del 10 % (redondeado hacia abajo, en guaraníes enteros).
- Dos `Radiobutton` (que comparten un `StringVar`) eligen "Retiro en local" o "Entrega a domicilio".
- La dirección solo se habilita y es obligatoria para entrega a domicilio.
- La entrega a domicilio suma G. 10.000 de envío, que se agrega **después** del descuento.
- El resumen muestra modalidad, dirección (si corresponde), subtotal, descuento, envío y total final.
- "Limpiar" pide confirmación y, si se acepta, vuelve a "Retiro en local", deja la dirección vacía y desmarca el descuento.
- Enter ejecuta el mismo cálculo que el botón.

## Reglas de validación

| Campo | Regla |
|---|---|
| Nombre | No vacío después de `strip()` |
| Teléfono | Entre 6 y 15 dígitos ASCII, sin espacios ni signos |
| Producto | Elegido del catálogo |
| Precio | Entero positivo, sin puntos ni comas |
| Cantidad | Entero entre 1 y 100 |
| Dirección | Obligatoria solo si la modalidad es domicilio |

## Tabla de pruebas

Probé el programa en la ventana y el comportamiento coincidió con lo esperado en todos los casos.

| Dato ingresado | Resultado esperado | Resultado observado |
|---|---|---|
| Ana; 0981123456; Gasas; 15000; 3; con descuento; domicilio con dirección | Total final: G. 50.500 | G. 50.500 |
| Mismo pedido, sin descuento, retiro en local | Total final: G. 45.000 | G. 45.000 |
| Mismo pedido, con descuento, retiro en local | Total final: G. 40.500 | G. 40.500 |
| Domicilio con dirección vacía | Aviso pidiendo la dirección; conserva los datos | Apareció el aviso y se conservaron los datos |
| Retiro en local con dirección vacía | Cálculo correcto sin exigir dirección | Calculó sin pedir dirección |
| Nombre solo con espacios | Aviso; conserva las entradas | Apareció el aviso y se conservaron las entradas |
| Teléfono `abc` | Aviso sobre el teléfono | Apareció el aviso sobre el teléfono |
| Producto sin elegir | Solicita seleccionar un producto | Solicitó seleccionar un producto |
| Precio `15.000` | Solicita un entero sin separadores | Solicitó un entero sin separadores |
| Cantidad `1` y `100` | Se aceptan | Se aceptaron |
| Cantidad `0`, `101`, `dos` | Aviso; no se muestra total | Apareció el aviso y no se mostró total |
| Limpiar y responder "No" | Se conserva el formulario y el resumen | Se conservaron el formulario y el resumen |
| Limpiar y responder "Sí" | Campos vacíos, cantidad 1, retiro en local, descuento desmarcado, foco en nombre | Se restableció todo y el foco volvió al nombre |
| Elegir "Retiro en local" después de domicilio | El campo de dirección se deshabilita | La dirección se deshabilitó |
| Ampliar la ventana | Entradas y resumen aprovechan el espacio | Entradas y resumen se ampliaron |

## Modificación de distribución

Agregué la sección **"3. Modalidad de entrega"** como un `LabelFrame` dentro del contenedor `principal`. Para que la dirección crezca al ampliar la ventana, configuré `columnconfigure(1, weight=1)` en ese `LabelFrame` (el padre directo de la entrada) y usé `sticky="ew"` en la entrada. Las acciones pasaron a la fila 4 y el resumen a la fila 5, que es la que recibe el espacio vertical sobrante con `rowconfigure(5, weight=1)`.

## Regla de validación agregada

Si la modalidad es "Entrega a domicilio" y la dirección está vacía después de `strip()`, `leer_pedido()` lanza un `ValueError` con el mensaje "Para entrega a domicilio debes indicar la dirección." `calcular()` lo captura, muestra un aviso y detiene el cálculo sin borrar los datos ingresados. Con retiro en local la dirección no se exige y se ignora aunque tenga texto.


In [2]:
import tkinter as tk
from tkinter import ttk, messagebox

PRODUCTOS = ("Alcohol 70%", "Gasas", "Guantes", "Termómetro")
COSTO_ENVIO = 10000
MODALIDAD_RETIRO = "retiro"
MODALIDAD_DOMICILIO = "domicilio"


def moneda(valor):
    return "G. " + f"{valor:,}".replace(",", ".")


def calcular_totales(precio, cantidad, con_descuento, modalidad):
    """Devuelve (subtotal, descuento, envio, total).
    El descuento se aplica solo sobre los productos; el envío se suma después."""
    subtotal = precio * cantidad
    descuento = subtotal // 10 if con_descuento else 0
    envio = COSTO_ENVIO if modalidad == MODALIDAD_DOMICILIO else 0
    total = subtotal - descuento + envio
    return subtotal, descuento, envio, total


ventana = tk.Tk()
ventana.title("Farmacia – Pedido con entrega")
ventana.geometry("720x780")
ventana.minsize(600, 700)
ventana.columnconfigure(0, weight=1)
ventana.rowconfigure(0, weight=1)

estilo = ttk.Style(ventana)
estilo.configure("Titulo.TLabel", font=("Arial", 18, "bold"))
estilo.configure("Principal.TButton", font=("Arial", 11, "bold"), padding=(12, 8))

# ---------- Variables ----------
nombre_var = tk.StringVar(ventana)
telefono_var = tk.StringVar(ventana)
producto_var = tk.StringVar(ventana)
precio_var = tk.StringVar(ventana)
cantidad_var = tk.StringVar(ventana, value="1")
descuento_var = tk.BooleanVar(ventana, value=False)
modalidad_var = tk.StringVar(ventana, value=MODALIDAD_RETIRO)
direccion_var = tk.StringVar(ventana)
resumen_var = tk.StringVar(ventana, value="Completa los datos y pulsa Calcular pedido.")
estado_var = tk.StringVar(ventana, value="Listo para registrar un pedido.")

# ---------- Contenedor principal ----------
principal = ttk.Frame(ventana, padding=15)
principal.grid(row=0, column=0, sticky="nsew")
principal.columnconfigure(0, weight=1)
principal.rowconfigure(5, weight=1)
ttk.Label(principal, text="Nuevo pedido", style="Titulo.TLabel").grid(
    row=0, column=0, sticky="w", pady=(0, 12))

# ---------- 1. Cliente ----------
cliente = ttk.LabelFrame(principal, text="1. Datos del cliente", padding=10)
cliente.grid(row=1, column=0, sticky="ew", pady=5)
cliente.columnconfigure(1, weight=1)
ttk.Label(cliente, text="Nombre:").grid(row=0, column=0, sticky="w", padx=5, pady=5)
entrada_nombre = ttk.Entry(cliente, textvariable=nombre_var)
entrada_nombre.grid(row=0, column=1, sticky="ew", padx=5, pady=5)
ttk.Label(cliente, text="Teléfono:").grid(row=1, column=0, sticky="w", padx=5, pady=5)
entrada_telefono = ttk.Entry(cliente, textvariable=telefono_var)
entrada_telefono.grid(row=1, column=1, sticky="ew", padx=5, pady=5)

# ---------- 2. Producto ----------
producto = ttk.LabelFrame(principal, text="2. Datos del producto", padding=10)
producto.grid(row=2, column=0, sticky="ew", pady=5)
producto.columnconfigure(1, weight=1)
ttk.Label(producto, text="Producto:").grid(row=0, column=0, sticky="w", padx=5, pady=5)
combo = ttk.Combobox(producto, textvariable=producto_var, values=PRODUCTOS, state="readonly")
combo.grid(row=0, column=1, sticky="ew", padx=5, pady=5)
ttk.Label(producto, text="Precio unitario (Gs.):").grid(row=1, column=0, sticky="w", padx=5, pady=5)
entrada_precio = ttk.Entry(producto, textvariable=precio_var)
entrada_precio.grid(row=1, column=1, sticky="ew", padx=5, pady=5)
ttk.Label(producto, text="Cantidad (1 a 100):").grid(row=2, column=0, sticky="w", padx=5, pady=5)
entrada_cantidad = ttk.Entry(producto, textvariable=cantidad_var)
entrada_cantidad.grid(row=2, column=1, sticky="ew", padx=5, pady=5)
ttk.Checkbutton(producto, text="Aplicar descuento del 10 %", variable=descuento_var).grid(
    row=3, column=0, columnspan=2, sticky="w", padx=5, pady=5)
ttk.Label(producto, text="Ejemplo de precio: 15000, sin puntos ni comas.").grid(
    row=4, column=0, columnspan=2, sticky="w", padx=5, pady=5)

# ---------- 3. Entrega ----------
entrega = ttk.LabelFrame(principal, text="3. Modalidad de entrega", padding=10)
entrega.grid(row=3, column=0, sticky="ew", pady=5)
entrega.columnconfigure(1, weight=1)


def actualizar_direccion():
    """Habilita la dirección solo para entrega a domicilio."""
    if modalidad_var.get() == MODALIDAD_DOMICILIO:
        entrada_direccion.configure(state="normal")
        entrada_direccion.focus_set()
    else:
        entrada_direccion.configure(state="disabled")


ttk.Radiobutton(entrega, text="Retiro en local", variable=modalidad_var,
                value=MODALIDAD_RETIRO, command=actualizar_direccion).grid(
    row=0, column=0, sticky="w", padx=5, pady=5)
ttk.Radiobutton(entrega, text=f"Entrega a domicilio (+ {moneda(COSTO_ENVIO)})",
                variable=modalidad_var, value=MODALIDAD_DOMICILIO,
                command=actualizar_direccion).grid(
    row=0, column=1, sticky="w", padx=5, pady=5)
ttk.Label(entrega, text="Dirección:").grid(row=1, column=0, sticky="w", padx=5, pady=5)
entrada_direccion = ttk.Entry(entrega, textvariable=direccion_var, state="disabled")
entrada_direccion.grid(row=1, column=1, sticky="ew", padx=5, pady=5)


# ---------- Validación ----------
def leer_pedido():
    nombre = nombre_var.get().strip()
    telefono = telefono_var.get().strip()
    if not nombre:
        raise ValueError("Completa el nombre del cliente.")
    if not (telefono.isascii() and telefono.isdigit() and 6 <= len(telefono) <= 15):
        raise ValueError("El teléfono debe contener entre 6 y 15 dígitos, sin signos ni espacios.")
    elegido = producto_var.get()
    if elegido not in PRODUCTOS:
        raise ValueError("Selecciona un producto del catálogo.")
    try:
        precio = int(precio_var.get().strip())
    except ValueError:
        raise ValueError("El precio debe ser un entero, por ejemplo 15000.")
    if precio <= 0:
        raise ValueError("El precio debe ser mayor que cero.")
    try:
        cantidad = int(cantidad_var.get().strip())
    except ValueError:
        raise ValueError("La cantidad debe ser un entero, por ejemplo 3.")
    if not (1 <= cantidad <= 100):
        raise ValueError("La cantidad debe estar entre 1 y 100.")
    modalidad = modalidad_var.get()
    direccion = direccion_var.get().strip()
    if modalidad == MODALIDAD_DOMICILIO and not direccion:
        raise ValueError("Para entrega a domicilio debes indicar la dirección.")
    if modalidad == MODALIDAD_RETIRO:
        direccion = ""
    return nombre, telefono, elegido, precio, cantidad, modalidad, direccion


# ---------- Acciones ----------
def calcular(event=None):
    resumen_var.set("Pedido pendiente de validación.")
    try:
        (nombre, telefono, elegido, precio, cantidad,
         modalidad, direccion) = leer_pedido()
    except ValueError as error:
        estado_var.set("Revisa los datos del pedido.")
        messagebox.showwarning("Datos incompletos o incorrectos", str(error), parent=ventana)
        return
    subtotal, descuento, envio, total = calcular_totales(
        precio, cantidad, descuento_var.get(), modalidad)
    lineas = [
        f"Cliente: {nombre}",
        f"Teléfono: {telefono}",
        f"Producto: {elegido}",
        f"Precio unitario: {moneda(precio)}",
        f"Cantidad: {cantidad}",
    ]
    if modalidad == MODALIDAD_DOMICILIO:
        lineas.append("Modalidad: Entrega a domicilio")
        lineas.append(f"Dirección: {direccion}")
    else:
        lineas.append("Modalidad: Retiro en local")
    lineas += [
        f"Subtotal: {moneda(subtotal)}",
        f"Descuento: {moneda(descuento)}",
        f"Envío: {moneda(envio)}",
        f"Total final: {moneda(total)}",
    ]
    resumen_var.set("\n".join(lineas))
    estado_var.set("Cálculo correcto. El pedido todavía no se guarda en un archivo.")


def limpiar():
    if not messagebox.askyesno("Limpiar formulario",
                               "¿Seguro que quieres borrar todos los datos?",
                               parent=ventana):
        return
    nombre_var.set("")
    telefono_var.set("")
    producto_var.set("")
    precio_var.set("")
    cantidad_var.set("1")
    descuento_var.set(False)
    modalidad_var.set(MODALIDAD_RETIRO)
    direccion_var.set("")
    actualizar_direccion()
    resumen_var.set("Completa los datos y pulsa Calcular pedido.")
    estado_var.set("Formulario limpio.")
    entrada_nombre.focus_set()


acciones = ttk.Frame(principal)
acciones.grid(row=4, column=0, sticky="ew", pady=10)
ttk.Button(acciones, text="Calcular pedido", command=calcular,
           style="Principal.TButton").grid(row=0, column=0, padx=(0, 8))
ttk.Button(acciones, text="Limpiar", command=limpiar).grid(row=0, column=1)

# ---------- 4. Resumen ----------
resumen = ttk.LabelFrame(principal, text="4. Resumen", padding=10)
resumen.grid(row=5, column=0, sticky="nsew", pady=5)
resumen.columnconfigure(0, weight=1)
resumen.rowconfigure(0, weight=1)
etiqueta_resumen = ttk.Label(resumen, textvariable=resumen_var, anchor="nw",
                             justify="left", wraplength=500)
etiqueta_resumen.grid(row=0, column=0, sticky="nsew")
etiqueta_resumen.bind(
    "<Configure>",
    lambda evento: etiqueta_resumen.configure(wraplength=max(100, evento.width)))
ttk.Label(principal, textvariable=estado_var, wraplength=530).grid(
    row=6, column=0, sticky="w", pady=(5, 0))

ventana.bind("<Return>", calcular)
entrada_nombre.focus_set()
ventana.mainloop()